# Create Sliding Windows

The method must only use flow and chlorine concentration sensors. So we do not use:

Pressure

Scenario

InjectionNode

InjectionAmount

StartDay

EndDay

Only

flow-columns and Chlorine-columns

In [3]:
import pandas as pd
import numpy as np

#########################################################
# LOAD DATASET
#########################################################

df = pd.read_csv("../data/dataset_hanoi.csv")

print("=" * 60)
print("Original Dataset Shape:", df.shape)
print("=" * 60)

#########################################################
# WINDOW SIZE
#########################################################

WINDOW_SIZE = 12        # Change to 4, 8, 12, 24 whenever you want

#########################################################
# SELECT FEATURES
#########################################################

flow_columns = [
    c for c in df.columns
    if "flow" in c.lower()
]

chlorine_columns = [
    c for c in df.columns
    if "chlorine" in c.lower()
]

feature_columns = flow_columns + chlorine_columns

print("\nNumber of Flow Features :", len(flow_columns))
print("Number of Chlorine Features :", len(chlorine_columns))
print("Total Features :", len(feature_columns))

#########################################################
# CREATE SLIDING WINDOWS
#########################################################

X = []
y = []
metadata = []

for scenario in df["Scenario"].unique():

    scenario_df = (
        df[df["Scenario"] == scenario]
        .reset_index(drop=True)
    )

    features = scenario_df[feature_columns]
    labels = scenario_df["Attack"]

    for i in range(WINDOW_SIZE - 1, len(scenario_df)):

        # Sliding Window
        window = features.iloc[
            i-WINDOW_SIZE+1 : i+1
        ].values.flatten()

        X.append(window)

        # Label
        y.append(labels.iloc[i])

        # Store metadata
        metadata.append({

            "Scenario": scenario,

            "InjectionNode":
                scenario_df.loc[i, "InjectionNode"],

            "InjectionAmount":
                scenario_df.loc[i, "InjectionAmount"]

        })

#########################################################
# CONVERT TO NUMPY
#########################################################

X = np.array(X)
y = np.array(y)

print("\nSliding Window Shape:", X.shape)

#########################################################
# CREATE COLUMN NAMES
#########################################################

column_names = []

for t in range(WINDOW_SIZE):

    if t == WINDOW_SIZE - 1:
        suffix = "_t"

    else:
        suffix = f"_t-{WINDOW_SIZE-1-t}"

    for feature in feature_columns:

        column_names.append(feature + suffix)

#########################################################
# CREATE DATAFRAME
#########################################################

window_df = pd.DataFrame(
    X,
    columns=column_names
)

#########################################################
# ADD METADATA
#########################################################

metadata_df = pd.DataFrame(metadata)

window_df = pd.concat(
    [metadata_df, window_df],
    axis=1
)

#########################################################
# ADD LABEL
#########################################################

window_df["Attack"] = y

#########################################################
# SAVE DATASET
#########################################################

filename = f"../data/dataset_hanoi_window{WINDOW_SIZE}.csv"

window_df.to_csv(
    filename,
    index=False
)

#########################################################
# SUMMARY
#########################################################

print("\n")
print("=" * 60)
print("SLIDING WINDOW DATASET CREATED")
print("=" * 60)

print("Window Size :", WINDOW_SIZE)
print("Rows :", len(window_df))
print("Columns :", len(window_df.columns))

print("\nAttack Distribution")
print(window_df["Attack"].value_counts())

print("\nSaved as:")
print(filename)

print("\nFirst Five Rows")
print(window_df.head())

print("=" * 60)


Original Dataset Shape: (51005, 15)

Number of Flow Features : 0
Number of Chlorine Features : 9
Total Features : 9

Sliding Window Shape: (49905, 108)


SLIDING WINDOW DATASET CREATED
Window Size : 12
Rows : 49905
Columns : 112

Attack Distribution
Attack
0    45821
1     4084
Name: count, dtype: int64

Saved as:
../data/dataset_hanoi_window12.csv

First Five Rows
   Scenario  InjectionNode  InjectionAmount  \
0         1             32       189.764465   
1         1             32       189.764465   
2         1             32       189.764465   
3         1             32       189.764465   
4         1             32       189.764465   

   bulk_species_node [MG] at Chlorine @ 3_t-11  \
0                                     0.000000   
1                                     0.003009   
2                                     0.003009   
3                                     0.003009   
4                                     0.003009   

   bulk_species_node [MG] at Chlorine @ 8_t-11  